# Imports


In [32]:
import numpy as np
import torch
import time
import copy
import torch
import rendering
import heightfield
import image
import opt
import sys
import simannealing
from datetime import datetime

In [33]:
print("Running with " + str(sys.version))

Running with 3.8.11 (default, Aug  6 2021, 08:56:27) 
[Clang 10.0.0 ]


# Grad descent

In [34]:
def grad_descent(num_epochs, H, COL, cam_directions, image_paths, use_barrier_loss = True, use_neighbor_loss = False, sim_annealing = True, optim_type = "Adam", heaviside_fn = None, heaviside_k = 0.1, num_repeats = 2, num_rays = 32):
  
  now = datetime.now()
  date_time = now.strftime("%m_%d_%Y_%H-%M-%S")
  num_cams = len(cam_directions)
  num_strips = H.shape[0]*num_repeats
  ray_width = 1/num_rays
  
  original_ray_origins = [[], [], [], []]
  ray_origins = copy.deepcopy(original_ray_origins)
  coords = []
  
  schedule = torch.zeros(50) #update colors for 10 steps, then heights for 20 steps
  schedule[10:20] = 1
  schedule[40:50] = 1

  grad_types = schedule.repeat(20)
  

  for i in range(num_rays):
    for k in range(num_rays):
      coord = np.array([i*ray_width + ray_width/2, k*ray_width + ray_width/2, 1])
      coords.append(coord)

      for q in range(num_cams):   
        curr_origin = coord - 20*cam_directions[q]
        curr_origin[2] = curr_origin[2].item()
        ray_origins[q].append(curr_origin)



  H = torch.tensor(H, dtype=torch.float, requires_grad = True)
  COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)



  ray_packs = []


  for t in range(num_cams):
    ray_packs.append(rendering.get_ray_packs(ray_origins[t], cam_directions[t], num_strips))

  
  desired_cols = []
  for i in range(num_cams):
    desired_cols.append(image.get_tensor_from_image(image_paths[i], (64, 64), (num_rays, num_rays)))
  
  
  H, COL = simannealing.sim_annealing(H, COL, ray_packs, num_cams, num_repeats, heaviside_fn, heaviside_k, desired_cols)
      
  if optim_type == "Adam":
    optimizer = torch.optim.AdamW([H, COL],lr=0.1)
    
  if optim_type == "SGD":
    optimizer = torch.optim.SGD([H, COL],lr=0.1, momentum=0.9)


  for j in range(1, num_epochs):
    
    
    torch.save(H, 'models/' + date_time + '_heights.pt')
    torch.save(COL, 'models/' + date_time + '_colors.pt')
    print("saving")
      
    change = False
    
    if j%100 == 0:
      H = heightfield.double_resolution(H)
      COL = heightfield.double_resolution(COL)
      H = torch.tensor(H, dtype=torch.float)
      COL = torch.tensor(COL, dtype=torch.float)
      num_strips = num_strips*2
      
      desired_cols = []
      for q in range(num_cams):

        desired_cols.append(image.get_tensor_from_image(image_paths[q], (64, 64), (num_rays, num_rays)))
      change = True


    if j%100 == 95:
      num_rays = num_rays*2
      ray_width = ray_width/2
      change = True
    
    if j%100 == 95 or j%100 == 0:
      coords = []
      change = True
      

      
      ray_origins = copy.deepcopy(original_ray_origins)
      for i in range(num_rays):
        for k in range(num_rays):
          coord = np.array([i*ray_width + ray_width/2, k*ray_width + ray_width/2, 1])
          coords.append(coord)
          for q in range(num_cams):    
            curr_origin = coord - 20*cam_directions[q] 
            curr_origin[2] = curr_origin[2].item()
            ray_origins[q].append(curr_origin)


      
      ray_packs = []
      for t in range(num_cams):
        ray_packs.append(rendering.get_ray_packs(ray_origins[t], cam_directions[t], num_strips))
        
      H.requires_grad = True
      COL.requires_grad = True
      
      desired_cols = []
      for i in range(num_cams):
        desired_cols.append(image.get_tensor_from_image(image_paths[i], (64, 64), (num_rays, num_rays)))

    if int(grad_types[j].item()) == 0:
      
      H.requires_grad = False
      COL.requires_grad = True
      

      lr = 0.05
      

    elif int(grad_types[j].item()) == 1:
      H.requires_grad = True
      COL.requires_grad = False 
      COL = COL.detach().numpy()
      COL = torch.tensor(COL, dtype=torch.float, requires_grad = False)
      if j > 49:
        lr = 0.03
      
    elif int(grad_types[j].item()) == 2:
      H.requires_grad = True
      COL.requires_grad = True
      H = torch.tensor(H, dtype=torch.float, requires_grad = True)
      COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)

    if j > 0:
      if int(grad_types[j].item()) != int(grad_types[j-1].item()):
        
        change = True
    if change:



      if optim_type == "Adam":
        optimizer = torch.optim.AdamW([H, COL],lr=lr)
      if optim_type == "SGD":
        optimizer = torch.optim.SGD([H, COL],lr=lr, momentum=0.9)




    result_cols = rendering.forward(H, COL, ray_packs, num_cams, num_repeats=num_repeats, heaviside_fn = heaviside_fn, heaviside_k = heaviside_k)

    mse_loss = 0
    for q in range(num_cams):
      mse_loss += torch.nn.functional.mse_loss(result_cols[q].double(), desired_cols[q].double())




    if j % 1 == 0 or j == 99:
      print("EPOCH: " + str(j) + "\t LOSS: " + str(mse_loss.item()/2))
      time.sleep(1)
    if use_barrier_loss:
      barrier_loss = opt.barrier_fn(H, min_val = 0.1, max_val = 1.5)
      mse_loss = mse_loss + 0.02*barrier_loss

    if use_neighbor_loss:
      neighbor_loss = opt.neighbor_fn(H)
      mse_loss = mse_loss + 0.5*neighbor_loss



      
       
    
    optimizer.zero_grad()
    mse_loss.backward()
    optimizer.step()



  repeated_H = H.repeat(num_repeats, num_repeats)
  repeated_COL = COL.repeat(num_repeats, num_repeats, 1)


  return repeated_H, repeated_COL, result_cols, desired_cols



# Main

In [35]:
#azimuth compatibility ablation

H = heightfield.get_uniform_H(4, 4, center =0.85)
COL = heightfield.get_white_COL(4)

cam_direction_1 = np.array([1, 0.000001, -1])

cos = [1, 0.94, 0.766, 0.5, 0.174, -0.174, -0.5, -0.766, -0.94, -1]
sin = [0.001, 0.34, 0.643, 0.866, 0.984, 0.984, 0.866, 0.643, 0.34, 0.001]

np.set_printoptions(threshold=1000)
torch.set_printoptions(threshold = 1000)

cam_direction_1 = np.array([1, 0.0001, -1])
cam_direction_2 = np.array([1, 0.0001, -10])
cam_direction_3 = np.array([-1, 0.00001, -1])

cam_directions = np.array([cam_direction_1, cam_direction_2, cam_direction_3])

num_epochs = 260
p1 = "test_images/1.png"
p2 = "test_images/2.png"
p3 = "test_images/3.png"
image_paths = [p1, p2, p3]
H1, COL1, result_cols, desired_cols = grad_descent(num_epochs, H, COL, cam_directions, image_paths, use_barrier_loss = True, use_neighbor_loss = False, optim_type="Adam", heaviside_fn = "tan", num_rays = 16)




/var/folders/1y/91nvhzqn12d2sh6dm1wf4_lh0000gn/T/ipykernel_3770/3607151942.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  H = torch.tensor(H, dtype=torch.float, requires_grad = True)
/var/folders/1y/91nvhzqn12d2sh6dm1wf4_lh0000gn/T/ipykernel_3770/3607151942.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)


STEP: 0	 LOSS: 0.9824497244548129
STEP: 1	 LOSS: 0.9757950542982987
STEP: 2	 LOSS: 0.9693693846591769
STEP: 3	 LOSS: 0.964938639847702
STEP: 4	 LOSS: 0.9624881275394566
STEP: 5	 LOSS: 0.9599983847156695
STEP: 6	 LOSS: 0.9572594975789195
STEP: 7	 LOSS: 0.9532890655410604
STEP: 8	 LOSS: 0.953824837984147
STEP: 9	 LOSS: 0.9526455543432739
STEP: 10	 LOSS: 0.9473346987419768
STEP: 11	 LOSS: 0.9520338634323302
STEP: 12	 LOSS: 0.9457952198845352
STEP: 13	 LOSS: 0.9398752185280521
STEP: 14	 LOSS: 0.9402321350852227
STEP: 15	 LOSS: 0.939730275411015
STEP: 16	 LOSS: 0.9408474431837268
STEP: 17	 LOSS: 0.9321947029408972
STEP: 18	 LOSS: 0.935572094057548
STEP: 19	 LOSS: 0.934029173153786
STEP: 20	 LOSS: 0.9243218720662985
STEP: 21	 LOSS: 0.9189501115242882
STEP: 22	 LOSS: 0.9101611306199788
STEP: 23	 LOSS: 0.908872670676603
STEP: 24	 LOSS: 0.9067639362201505
STEP: 25	 LOSS: 0.9069102932158195
STEP: 26	 LOSS: 0.9035303715255341
STEP: 27	 LOSS: 0.8969757478020863
STEP: 28	 LOSS: 0.8945221243268913
S

/var/folders/1y/91nvhzqn12d2sh6dm1wf4_lh0000gn/T/ipykernel_3770/3607151942.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  H = torch.tensor(H, dtype=torch.float)
/var/folders/1y/91nvhzqn12d2sh6dm1wf4_lh0000gn/T/ipykernel_3770/3607151942.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  COL = torch.tensor(COL, dtype=torch.float)


EPOCH: 100	 LOSS: 0.22375009142305788
saving
EPOCH: 101	 LOSS: 0.2111870194191135
saving
EPOCH: 102	 LOSS: 0.20274420285448871
saving
EPOCH: 103	 LOSS: 0.19831636709007192
saving
EPOCH: 104	 LOSS: 0.1960081774199908
saving
EPOCH: 105	 LOSS: 0.19429797230949283
saving
EPOCH: 106	 LOSS: 0.19274711089969196
saving
EPOCH: 107	 LOSS: 0.19132841482070204
saving
EPOCH: 108	 LOSS: 0.18996595553968965
saving
EPOCH: 109	 LOSS: 0.18847827830839067
saving
EPOCH: 110	 LOSS: 0.1868423919027958
saving
EPOCH: 111	 LOSS: 0.199660973691985
saving
EPOCH: 112	 LOSS: 0.1851900635475736
saving
EPOCH: 113	 LOSS: 0.1846009452257545
saving
EPOCH: 114	 LOSS: 0.18703993477514202
saving
EPOCH: 115	 LOSS: 0.18867874182874886
saving
EPOCH: 116	 LOSS: 0.1865323770550649
saving
EPOCH: 117	 LOSS: 0.1829780019002635
saving
EPOCH: 118	 LOSS: 0.18114184076980763
saving
EPOCH: 119	 LOSS: 0.18019632068592917
saving
EPOCH: 120	 LOSS: 0.180026179752712
saving
EPOCH: 121	 LOSS: 0.17369783771021727
saving
EPOCH: 122	 LOSS: 0.1